### **Direct Preference Optimization (DPO)**

In [ ]:
%pip install -q -U transformers datasets trl peft accelerate bitsandbytes sentencepiece
!pip uninstall -y torchao
!pip install -U "torchao>=0.16.0"
!pip install -q -U unsloth

In [ ]:
import gc
import json
import shutil
import os
import random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import transformers
import datasets
import trl
import peft
from datasets import Dataset, load_dataset
from peft import LoraConfig, AutoPeftModelForCausalLM
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from trl import DPOConfig, DPOTrainer
from trl.experimental.orpo import ORPOConfig, ORPOTrainer
from unsloth import (
    FastLanguageModel,
    PatchDPOTrainer,
    is_bfloat16_supported,
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)



> Dataset options:
   1. "inline"        -> uses the four examples included below
   2. "json"          -> loads a local JSON file containing prompt/chosen/rejected
   3. ultrafeedback" -> loads a subset of trl-lib/ultrafeedback_binarized



In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "outputs/qwen2_5_0_5b_dpo_lora"
DATA_SOURCE = "inline"
JSON_PATH = "preference_data.json"
ULTRAFEEDBACK_MAX_SAMPLES = 2_000
USE_4BIT_IF_CUDA = True
MAX_LENGTH = 512
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 5e-5
BETA = 0.1
INLINE_MAX_STEPS = 40

print("Model:", MODEL_NAME)
print("Dataset source:", DATA_SOURCE)
print("Output directory:", OUTPUT_DIR)

In [ ]:
HAS_CUDA = torch.cuda.is_available()
SUPPORTS_BF16 = HAS_CUDA and torch.cuda.is_bf16_supported()
USE_BF16 = SUPPORTS_BF16
USE_FP16 = HAS_CUDA and not SUPPORTS_BF16
USE_4BIT = HAS_CUDA and USE_4BIT_IF_CUDA

if USE_BF16:
    COMPUTE_DTYPE = torch.bfloat16
elif USE_FP16:
    COMPUTE_DTYPE = torch.float16
else:
    COMPUTE_DTYPE = torch.float32

if HAS_CUDA:
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: not available; CPU training will be slow.")

print("BF16:", USE_BF16)
print("FP16:", USE_FP16)
print("4-bit QLoRA:", USE_4BIT)
print("Compute dtype:", COMPUTE_DTYPE)

In [ ]:
INLINE_DATA = [
    {
        "prompt": "Explain what machine learning is in simple words.",
        "chosen": "Machine learning is a way to teach computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every case.",
        "rejected": "Machine learning is a machine that automatically learns everything without data.",
    },
    {
        "prompt": "What is overfitting in machine learning?",
        "chosen": "Overfitting happens when a model memorizes the training data too closely and performs poorly on new unseen data.",
        "rejected": "Overfitting means the model becomes perfect for all types of data.",
    },
    {
        "prompt": "What is the difference between classification and regression?",
        "chosen": "Classification predicts categories or labels, while regression predicts continuous numerical values.",
        "rejected": "Classification and regression are exactly the same thing.",
    },
    {
        "prompt": "What is the purpose of a loss function?",
        "chosen": "A loss function measures how far a model's predictions are from the true values, guiding the model during training.",
        "rejected": "A loss function increases model size and does not affect training.",
    },
]

def convert_plain_text_to_conversational(example):
  if isinstance(example.get("prompt"), str):
    return {
        "prompt": [{"role": "user", "content": example["prompt"]}],
        "chosen": [{"role": "assistant", "content": example["chosen"]}],
        "rejected": [{"role": "assistant", "content": example["rejected"]}],
    }

  return example

if DATA_SOURCE == "inline":
  dataset = Dataset.from_list(INLINE_DATA)
  dataset = dataset.map(convert_plain_text_to_conversational)
elif DATA_SOURCE == "json":
  json_path = Path(JSON_PATH)
  if not json_path.exists():
    raise FileNotFoundError(
        f"{JSON_PATH!r} was not found. Upload the file or change JSON_PATH."
    )

  dataset = load_dataset("json", data_files=str(json_path), split="train")
  required_columns = ["prompt", "chosen", "rejected"]
  missing = required_columns - set(dataset.column_names)
  if missing:
    raise ValueError(f"The JSON file is missing columns: {sorted(missing)}")
  dataset = dataset.map(convert_plain_text_to_conversational)
elif DATA_SOURCE == "ultrafeedback":
  dataset = load_dataset(
      "trl-lib/ultrafeedback_binarized",
      split="train",
  )
  dataset = dataset.shuffle(seed=SEED)
  simple_count = min(ULTRAFEEDBACK_MAX_SAMPLES, len(dataset))
  dataset = dataset.select(range(simple_count))
else:
  raise ValueError(
    "DATA_SOURCE must be 'inline', 'json', or 'ultrafeedback'."
  )

print(dataset)
print("\nColumns:", dataset.column_names)
print("\nFirst example:")
print(json.dumps(dataset[0], indent=2, ensure_ascii=False))

In [ ]:
required_columns = {"chosen", "rejected"}
missing_columns = required_columns - set(dataset.column_names)

if missing_columns:
  raise ValueError(
      f"DPO requires chosen and rejected columns. Missing: {missing_columns}"
  )

if len(dataset) < 2:
  raise ValueError("At least two preference examples are required.")

test_size = 1 if len(dataset) < 20 else 0.05
split_dataset = dataset.train_test_split(test_size=test_size, seed=SEED)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print("Training examples:", len(train_dataset))
print("Evaluation examples:", len(eval_dataset))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer.padding_side = "left"
quantization_config = None

if USE_4BIT:
  quantization_config = BitsAndBytesConfig(
      load_in_4bit=True,
      bnb_4bit_use_double_quant=True,
      bnb_4bit_quant_type="nf4",
      bnb_4bit_compute_dtype=COMPUTE_DTYPE,
  )

print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)
print("Padding side:", tokenizer.padding_side)
print("Quantization configuration:")
print(quantization_config)

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

model_init_kwargs = {
    "dtype": COMPUTE_DTYPE,
    "trust_remote_code": True,
    "attn_implementation": "sdpa",
}

is_tiny_inline_demo = DATA_SOURCE == "inline"

In [ ]:
training_args = DPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1.0,
    max_steps=INLINE_MAX_STEPS if is_tiny_inline_demo else -1,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit" if USE_4BIT else "adamw_torch",
    max_grad_norm=1.0,
    beta=BETA,
    loss_type="sigmoid",
    max_length=MAX_LENGTH,
    truncation_mode="keep_start",
    bf16=USE_BF16,
    fp16=USE_FP16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    use_cache=False,
    eval_strategy="steps",
    eval_steps=10,
    logging_steps=1,
    logging_first_step=True,
    save_strategy="steps",
    save_steps=20,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    model_init_kwargs=model_init_kwargs,
)

trainer = DPOTrainer(
    model=MODEL_NAME,
    ref_model=None,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
    quantization_config=quantization_config,
)

trainer.model.print_trainable_parameters()
initial_metrics = trainer.evaluate()
print("Initial evaluation metrics:")
for key, value in initial_metrics.items():
    print(f"{key}: {value}")

In [ ]:
train_result = trainer.train()

print("Training completed.")
print("Global steps:", trainer.state.global_step)
print("Training loss:", train_result.training_loss)

In [ ]:
final_metrics = trainer.evaluate()

print("Final evaluation metrics:")
for key, value in final_metrics.items():
    print(f"{key}: {value}")

In [ ]:
history = pd.DataFrame(trainer.state.log_history)
display(history.tail(20))

loss_history = history.dropna(subset=["loss"]) if "loss" in history.columns else pd.DataFrame()
if not loss_history.empty:
    ax = loss_history.plot(
        x="step",
        y="loss",
        title="DPO Training Loss",
        grid=True,
        figsize=(8, 4),
    )
    ax.set_ylabel("Loss")

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Saved DPO LoRA adapter and tokenizer to: {OUTPUT_DIR}")
print("Saved files:")
for file_name in sorted(os.listdir(OUTPUT_DIR)):
    print(" -", file_name)

In [ ]:
del trainer
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
inference_load_kwargs = {
    "torch_dtype": COMPUTE_DTYPE,
}

if HAS_CUDA:
  inference_load_kwargs["device_map"] = "auto"
if quantization_config is not None:
  inference_load_kwargs["quantization_config"] = quantization_config

model = AutoPeftModelForCausalLM.from_pretrained(
    OUTPUT_DIR,
    **inference_load_kwargs,
)
model.eval()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    OUTPUT_DIR,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

def generate_response(prompt: str, max_new_tokens: int = 150, temperature: float = 0.7, top_p: float = 0.9) -> str:
  messages = [{"role": "user", "content": prompt}]
  formatted_prompt = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True,
  )

  inputs = tokenizer(
      formatted_prompt,
      return_tensors="pt",
  ).to(model.device)

  prompt_token_count = inputs["input_ids"].shape[1]
  generation_kwargs = {
      "max_new_tokens": max_new_tokens,
      "pad_token_id": tokenizer.pad_token_id,
      "eos_token_id": tokenizer.eos_token_id,
  }

  if temperature > 0:
    generation_kwargs.update({
      "do_sample": True,
      "temperature": temperature,
      "top_p": top_p,
    })
  else:
    generation_kwargs["do_sample"] = False

  with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        **generation_kwargs,
    )

  new_tokens = outputs[0, prompt_token_count:]
  return tokenizer.decode(
      new_tokens,
      skip_special_tokens=True,
  ).strip()

prompt = "Explain overfitting in machine learning in simple words."
response = generate_response(prompt)

print("Prompt:")
print(prompt)
print("\nDPO response:")
print(response)

In [ ]:
test_prompts = [
    "Explain machine learning in simple words.",
    "What is overfitting?",
    "How are classification and regression different?",
    "What does a loss function do?",
]

for current_prompt in test_prompts:
    print("=" * 80)
    print("PROMPT:", current_prompt)
    print("RESPONSE:", generate_response(current_prompt, temperature=0.3))

In [ ]:
archive_path = shutil.make_archive(
    base_name="qwen2_5_0_5b_dpo_lora",
    format="zip",
    root_dir=OUTPUT_DIR,
)

print("Created:", archive_path)

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "outputs/dpo_model"
dataset = load_dataset("trl-lib/ultrafeedback_binarized", split="train")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

peft_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
)

In [ ]:
training_args = DPOConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    num_train_epochs=3,
    logging_steps=1,
    save_steps=20,
    save_total_limit=2,
    bf16=torch.cuda.is_available(),
    fp16=not torch.cuda.is_available(),
    gradient_checkpointing=True,
    report_to="none",
    max_length=512,
    beta=0.1,
    loss_type="sigmoid"
)

trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

trainer.train()

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

In [ ]:
MODEL_PATH = "outputs/dpo_model_2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

prompt = "Explain overfitting in machine learning."
messages = [
    {"role": "user", "content": prompt}
]

try:
  text = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True,
  )
except Exception:
  text = prompt

inputs = tokenizer(text, return_tensors="pt").to(model.device)
with torch.no_grad():
  outputs = model.generate(
      **inputs,
      max_new_tokens=150,
      do_sample=True,
      temperature=0.7,
      top_p=0.9,
  )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

### **Odd Ratio Preferenec Optimization(ORPO)**

In [ ]:
model_name = "Qwen/Qwen2-0.5B-Instruct"
dataset_name = "trl-lib/ultrafeedback_binarized"
output_dir = "./qwen2_orpo_demo"

In [ ]:
use_cuda = torch.cuda.is_available()
use_bf16 = (
    use_cuda
    and torch.cuda.is_bf16_supported()
)
use_fp16 = (
    use_cuda
    and not use_bf16
)

if use_bf16:
    compute_dtype = torch.bfloat16
elif use_fp16:
    compute_dtype = torch.float16
else:
    compute_dtype = torch.float32
print(f"CUDA available: {use_cuda}")
print(f"BF16 training: {use_bf16}")
print(f"FP16 training: {use_fp16}")
print(f"Model dtype: {compute_dtype}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=compute_dtype,
)

model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id

dataset = load_dataset(
    dataset_name,
    split="train",
)

dataset = dataset.shuffle(seed=42)
number_of_samples = min(100, len(dataset))
dataset = dataset.select(
    range(number_of_samples)
)

print("Dataset columns:", dataset.column_names)
print("Number of training samples:", len(dataset))
print("First sample:", dataset[0])

In [ ]:
dataset = dataset.select(
    range(min(100, len(dataset)))
)

orpo_args = ORPOConfig(
    output_dir=output_dir,
    learning_rate=8e-6,
    beta=0.1,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    bf16=use_bf16,
    fp16=use_fp16,
    gradient_checkpointing=True,
    max_length=512,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    seed=42,
)

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)

trainer = ORPOTrainer(
    model=model,
    args=orpo_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

train_result = trainer.train()

In [ ]:
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

In [ ]:
trained_model = trainer.model
trained_model.eval()
trained_model.config.use_cache = True

prompt = "Explain reinforcement learning in simple terms."
messages = [
    {
        "role": "user",
        "content": prompt,
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(
    text,
    return_tensors="pt",
).to(trained_model.device)

with torch.no_grad():
    outputs = trained_model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

input_length = inputs["input_ids"].shape[1]
generated_tokens = outputs[0][input_length:]
response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True,
)

print("Model response:")
print(response)

In [ ]:
model_name = "Qwen/Qwen2-0.5B-Instruct"
dataset_name = "trl-lib/ultrafeedback_binarized"
output_dir = "./qwen2_orpo_unsloth"

max_seq_length = 512
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=load_in_4bit,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"
model.config.use_cache = False

In [ ]:
model = FastLanguageModel.get_peft_model(
    r=8,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

dataset = load_dataset(dataset_name, split="train")
dataset = dataset.shuffle(seed=42)
number_of_samples = min(500, len(dataset))
dataset = dataset.select(
    range(number_of_samples)
)

print("Original columns:", dataset.column_names)
print("Number of samples:", len(dataset))
print(dataset[0])

In [ ]:
def format_preference_example(example):
    chosen_messages = example["chosen"]
    rejected_messages = example["rejected"]

    if len(chosen_messages) < 2 or len(rejected_messages) < 2:
        raise ValueError(
            "Each chosen/rejected conversation must contain "
            "a prompt and an assistant response."
        )

    prompt_messages = chosen_messages[:-1]
    chosen_response = chosen_messages[-1]["content"]
    rejected_response = rejected_messages[-1]["content"]
    formatted_prompt = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    return {
        "prompt": formatted_prompt,
        "chosen": chosen_response + tokenizer.eos_token,
        "rejected": rejected_response + tokenizer.eos_token,
    }

dataset = dataset.map(
    format_preference_example,
    remove_columns=dataset.column_names,
)

print("Formatted columns:", dataset.column_names)
print("\nPrompt:\n", dataset[0]["prompt"])
print("\nChosen:\n", dataset[0]["chosen"][:500])
print("\nRejected:\n", dataset[0]["rejected"][:500])

In [ ]:
PatchDPOTrainer()

In [ ]:
orpo_args = ORPOConfig(
    output_dir=output_dir,
    learning_rate=8e-6,
    beta=0.1,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_length=max_seq_length,
    max_prompt_length=max_seq_length // 2,
    max_completion_length=max_seq_length // 2,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",
    lr_scheduler_type="linear",
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=1,
    remove_unused_columns=False,
    report_to="none",
    seed=42,
)

trainer = ORPOTrainer(
    model=model,
    args=orpo_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

train_result = trainer.train()
print(train_result)

In [ ]:
FastLanguageModel.for_inference(model)
model.config.use_cache = True
model.eval()

prompt = "Explain reinforcement learning in simple terms."
messages = [
    {
        "role": "user",
        "content": prompt,
    }
]

input_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        input_ids=input_ids,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

generated_ids = output_ids[0][input_ids.shape[1]:]
response = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True,
)

print("Model response:")
print(response)

In [ ]:
merged_output_dir = "./qwen2_orpo_merged_16bit"
model.save_pretrained_merged(
    merged_output_dir,
    tokenizer=tokenizer,
    save_method="merged_16bit",
)

print(f"Merged model saved at: {merged_output_dir}")